# CrediX Dual-Engine Fraud Detection Model — Training & Validation Notebook

**Version:** v3.0.0-monotonic-production  
**Framework:** scikit-learn (Multi-Tree Isolation Forest + Monotonic HistGradientBoosting)  
**Regulatory Compliance:** Basel III & Central Bank of Egypt Model Risk Management (MRM) Guidelines  

---

## 1. Executive Summary & Architecture

This notebook presents the training, monotonic mathematical calibration, and statistical stress testing of the **CrediX Dual-Engine Fraud Machine Learning Layer**.

### The 5-Layer Defense-in-Depth Architecture:
1. **Layer 1: Deterministic Cross-Document & Telemetry Rules:** NID format, salary slip mismatch, OCR quality, I-Score age, statement arithmetic balance check, VPN/proxy, off-hours.
2. **Layer 2: Deep Forensic Analysis:** Benford's Law Chi-Square distribution on transaction cashflows, Terminal-digit Chi-Square test, Micro-transaction absence analysis, Round number uniformity, and Adversarial threshold gaming.
3. **Layer 3: Real-Time Entity Velocity Store (SQLite):** 48-hour collision defense on NID, phone, bank account, and hardware device fingerprint.
4. **Layer 4: Dual-Engine Monotonic ML (This Notebook):** Unsupervised Isolation Forest (200 trees) + Monotonic Cost-Sensitive Gradient Boosting.
5. **Layer 5: Hybrid Decision Fusion & Bilingual Explainable AI (XAI):** Calibrated haircut factor and regulatory reason codes.

---
## 2. Dataset Governance & Realistic Covariance Structure

> ### 🛡️ MODEL GOVERNANCE & REAL-WORLD FIDELITY
> Unlike naive synthetic generators with independent draws, our population `data/fraud_training_data_25000.csv` models **realistic retail banking covariance**:
> - **Income Mismatch vs OCR Quality:** Document alterations introduce image artifacts and font layer degradation ($r = -0.65$).
> - **Surge vs Volatility:** Window-dressed balances experience temporary spikes accompanied by sharp cashflow dispersion ($r = +0.72$).
> - **Authentic Human Friction:** 14% of clean borrowers experience camera glare/lighting issues, 18% deposit round cash at ATMs, and 12% have freelance income variance without being fraudulent.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
import joblib

print('Libraries and dependencies loaded successfully.')

---
## 3. Loading the Benchmark Population (25,000 Records)

In [ ]:
data_path = os.path.join('..', 'data', 'fraud_training_data_25000.csv')
if not os.path.exists(data_path):
    data_path = 'fraud_training_data_25000.csv'

df = pd.read_csv(data_path)
print(f'Total Applications: {len(df):,}')
print(f'Feature Columns:    {df.shape[1]}')
print('\nPopulation Breakdown by Class:')
print(df['record_type'].value_counts())

---
## 4. Feature Space & Monotonic Constraints Specification

In credit risk and fraud modeling, unconstrained tree models can make erratic decisions (e.g., higher income discrepancy accidentally lowering predicted risk due to random tree splits).

To enforce strict banking logic, we apply **Monotonic Constraints (`monotonic_cst`)**:
- `+1` (Strictly Increasing Risk): `income_mismatch_ratio`, `annuity_to_balance_ratio`, `balance_volatility_cv`, `surge_ratio_max_to_avg`, `inflow_uniformity_score`, `bureau_facilities_count`
- `-1` (Strictly Decreasing Risk): `ocr_quality_mean`, `min_to_avg_balance_ratio`, `employment_tenure_years`, `inflow_regularity_score`, `iscore_normalized`
- `0` (Neutral): `applicant_age_norm`

In [ ]:
FEATURE_NAMES = [
    'income_mismatch_ratio', 'annuity_to_balance_ratio', 'balance_volatility_cv',
    'surge_ratio_max_to_avg', 'ocr_quality_mean', 'min_to_avg_balance_ratio',
    'applicant_age_norm', 'employment_tenure_years', 'inflow_regularity_score',
    'iscore_normalized', 'inflow_uniformity_score', 'bureau_facilities_count'
]

MONOTONIC_CONSTRAINTS = [+1, +1, +1, +1, -1, -1, 0, -1, -1, -1, +1, +1]

print('Canonical 12 Features and Monotonic Directions:')
for feat, cst in zip(FEATURE_NAMES, MONOTONIC_CONSTRAINTS):
    direction = 'Increases Risk (+1)' if cst == 1 else ('Decreases Risk (-1)' if cst == -1 else 'Neutral (0)')
    print(f'  {feat:<28} -> {direction}')

---
## 5. 5-Fold Stratified Cross-Validation

Evaluating model stability across 5 independent folds to verify out-of-fold generalization.

In [ ]:
X = df[FEATURE_NAMES].values
y = df['is_fraud'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X[train_idx], y[train_idx]
    X_va, y_va = X[val_idx], y[val_idx]
    
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)
    
    weights = np.where(y_tr == 1, 7.0, 1.0)
    clf = HistGradientBoostingClassifier(
        max_iter=150, learning_rate=0.05, max_leaf_nodes=31,
        min_samples_leaf=35, l2_regularization=2.5,
        monotonic_cst=MONOTONIC_CONSTRAINTS, random_state=42 + fold
    )
    clf.fit(X_tr_s, y_tr, sample_weight=weights)
    pr = clf.predict_proba(X_va_s)[:, 1]
    auc = roc_auc_score(y_va, pr)
    cv_aucs.append(auc)
    print(f'Fold {fold+1} ROC-AUC: {auc:.4f}')

print(f'\nMean 5-Fold Cross-Validation ROC-AUC: {np.mean(cv_aucs):.4f}')

---
## 6. Training Final Dual-Engine & Artifact Export

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 1. Unsupervised Isolation Forest (200 trees)
iso_model = IsolationForest(n_estimators=200, contamination=0.055, max_samples=256, random_state=42, n_jobs=-1)
iso_model.fit(X_scaled)

# 2. Supervised Cost-Sensitive HistGB (7x penalty)
weights_full = np.where(y == 1, 7.0, 1.0)
gb_model = HistGradientBoostingClassifier(
    max_iter=150, learning_rate=0.05, max_leaf_nodes=31,
    min_samples_leaf=35, l2_regularization=2.5,
    monotonic_cst=MONOTONIC_CONSTRAINTS, random_state=42
)
gb_model.fit(X_scaled, y, sample_weight=weights_full)

out_dir = os.path.join('..', 'model', 'artifacts', 'fraud')
os.makedirs(out_dir, exist_ok=True)
joblib.dump(iso_model, os.path.join(out_dir, 'isolation_forest_v2.joblib'))
joblib.dump(gb_model, os.path.join(out_dir, 'fraud_gradient_boost_v2.joblib'))
joblib.dump(scaler, os.path.join(out_dir, 'scaler_v2.joblib'))
print('Production models trained and exported successfully.')

---
## 7. Independent Out-of-Distribution (OOD) Stress Test (3,000 Cases)

To prove real-world robustness, we evaluate the production model against a strictly independent holdout dataset of 3,000 applications generated with shifted distributions and adversarial edge cases.

In [ ]:
metrics_file = os.path.join(out_dir, 'metrics.json')
if os.path.exists(metrics_file):
    with open(metrics_file, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    print(json.dumps(metrics['independent_ood_stress_test'], indent=2))
else:
    print('Metrics file not found. Run model/train_fraud.py first.')